# XGBoost: Gradient-Boosted Tree Ensemble

This notebook implements XGBoost, a gradient-boosted tree ensemble with built-in shrinkage and regularization.

**Goal**: Balance bias reduction with variance control through boosting and regularization.

## Key Features of XGBoost:
- **Gradient Boosting**: Sequentially builds trees to correct errors
- **Shrinkage (Learning Rate)**: Controls how much each tree contributes
- **Regularization**: L1/L2 penalties on leaf weights
- **Tree Pruning**: max_depth controls complexity

## Evaluation Metrics:
- **F1-Score** (primary): Mean ± standard deviation across 5 folds
- **ROC-AUC**, **Precision**, **Recall**

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Import the evaluate_model function
from evaluate_model import evaluate_model

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load preprocessed training data
print("Loading preprocessed training data...")
train_df = pd.read_csv('../data/train_data_preprocessed.csv')
print(f"Training data shape: {train_df.shape}")

# Prepare features and target
X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue']

print(f"\nFeatures shape: {X.shape}")
print(f"Features: {list(X.columns)}")

In [ ]:
# Setup cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2. XGBoost with Hyperparameter Tuning

We'll tune key hyperparameters:
- **n_estimators**: Number of boosting rounds
- **learning_rate**: Shrinkage parameter (controls bias-variance tradeoff)
- **max_depth**: Maximum tree depth (controls complexity)
- **subsample**: Row sampling fraction (reduces variance)
- **colsample_bytree**: Column sampling fraction (reduces variance)
- **reg_alpha**: L1 regularization
- **reg_lambda**: L2 regularization

In [ ]:
# XGBoost classifier
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Hyperparameter grid
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [0.1, 1.0, 10.0]
}

print("Note: Full grid search would take a long time.")
print("Consider using RandomizedSearchCV for faster hyperparameter tuning.")

In [ ]:
# Use RandomizedSearchCV for faster tuning
from sklearn.model_selection import RandomizedSearchCV

random_search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_grid_xgb,
    n_iter=50,  # Try 50 random combinations
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

print("Training XGBoost with Randomized Search...")
print("This may take several minutes...")
random_search_xgb.fit(X, y)

print(f"\nBest parameters: {random_search_xgb.best_params_}")
print(f"Best F1-Score: {random_search_xgb.best_score_:.4f}")

In [ ]:
# Evaluate best XGBoost model
results_xgb = evaluate_model(random_search_xgb.best_estimator_, X, y, cv, "XGBoost")

## 3. Feature Importance Analysis

In [ ]:
# Get feature importances
import matplotlib.pyplot as plt

feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': random_search_xgb.best_estimator_.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importances.head(10))

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(feature_importances.head(10)['feature'], feature_importances.head(10)['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Learning Curves (Optional)

Visualize how performance improves with more boosting rounds.

In [ ]:
# Train model with evaluation set to see learning curve
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

xgb_eval = xgb.XGBClassifier(
    **random_search_xgb.best_params_,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

eval_set = [(X_train, y_train), (X_val, y_val)]
xgb_eval.fit(X_train, y_train, eval_set=eval_set, verbose=False)

# Get evaluation results
results = xgb_eval.evals_result()

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(results['validation_0']['logloss'], label='Train')
plt.plot(results['validation_1']['logloss'], label='Validation')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('XGBoost Learning Curves')
plt.legend()
plt.grid(True)
plt.show()

## 5. Summary

In [ ]:
print(f"\n{'='*60}")
print("XGBOOST MODEL SUMMARY")
print(f"{'='*60}")
print(f"\nBest Hyperparameters:")
for param, value in random_search_xgb.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nCross-Validation Performance:")
print(f"  F1-Score:  {results_xgb['cv_f1']:.4f}")
print(f"  Precision: {results_xgb['cv_precision']:.4f}")
print(f"  Recall:    {results_xgb['cv_recall']:.4f}")
print(f"  ROC-AUC:   {results_xgb['cv_roc_auc']:.4f}")

print(f"\n{'='*60}")
print("KEY INSIGHTS")
print(f"{'='*60}")
print(f"\nBias-Variance Tradeoff:")
print(f"  - learning_rate: {random_search_xgb.best_params_['learning_rate']} (shrinkage)")
print(f"  - max_depth: {random_search_xgb.best_params_['max_depth']} (controls complexity)")
print(f"  - subsample: {random_search_xgb.best_params_['subsample']} (reduces variance)")
print(f"  - reg_alpha: {random_search_xgb.best_params_['reg_alpha']} (L1 regularization)")
print(f"  - reg_lambda: {random_search_xgb.best_params_['reg_lambda']} (L2 regularization)")

print(f"\nStrengths:")
print(f"  - Built-in regularization prevents overfitting")
print(f"  - Gradient boosting reduces bias sequentially")
print(f"  - Handles nonlinear interactions automatically")
print(f"  - Often achieves state-of-the-art performance")

## 6. Comparison with Other Models

Compare XGBoost performance with:
- Baseline Logistic Regression
- Regularized Logistic Regression (L1/L2)
- Random Forest
- Neural Networks

**Next Steps:**
- Test neural networks for flexible nonlinear modeling
- Compare all models to select the best performer
- Evaluate final model on holdout set